### Práctica 1

Definimos las exploradoras

In [ ]:
from threading import Lock, Thread, Semaphore, Event
import time
import math


class CentralOperaciones:
    def __init__(self, seg_simulado=0.01):
        self.coordenadas_doradas = []
        self.coords_registradas = set()
        self.estado_pepitas = {}

        self.rutas_pendientes = []
        self.rutas_registradas = set()

        self.cerrojo_rutas = Lock()
        self.cerrojo_coords = Lock()
        self.cerrojo_monedero = Lock()
        self.cerrojo_exploracion = Lock()

        self.recolectoras_viajeras = []
        self.calculadoras_ruta = []
        self.exploradoras = []
        self.recolectoras = []

        self.num_pepitas = 0
        self.seg_simulado = seg_simulado

        self.ultimo_x = 0
        self.sem_carretillas = Semaphore(0)
        self.stop_event = Event()

    def _añadir_pepita(self, cantidad=1):
        with self.cerrojo_monedero:
            self.num_pepitas += cantidad

    def _quitar_pepitas(self, cantidad):
        with self.cerrojo_monedero:
            if self.num_pepitas >= cantidad:
                self.num_pepitas -= cantidad
                return True
            return False

    def _reactivar_pepita(self, coord):
        time.sleep(86400 * self.seg_simulado)
        if self.stop_event.is_set():
            return
        with self.cerrojo_coords:
            if coord in self.estado_pepitas:
                self.estado_pepitas[coord] = True

    def _añadir_coordenada_dorada(self, coord):
        if hash(coord) % 1000000 == 0:
            with self.cerrojo_coords:
                if coord not in self.coords_registradas:
                    self.coordenadas_doradas.append(coord)
                    self.coords_registradas.add(coord)
                    self.estado_pepitas[coord] = True

    def _exploradora(self):
        while not self.stop_event.is_set():
            with self.cerrojo_exploracion:
                x_a_explorar = self.ultimo_x
                self.ultimo_x += 1

            for y in range(-1000, 1000):
                if self.stop_event.is_set():
                    return
                self._añadir_coordenada_dorada((x_a_explorar, y))

            time.sleep(self.seg_simulado)

    def añadir_exploradora(self):
        t = Thread(target=self._exploradora, daemon=True)
        t.start()
        self.exploradoras.append(t)
        return t

    def _recolectora(self):
        print("Recolectora clásica lista en la base")

        while not self.stop_event.is_set():
            coord_obj = None

            with self.cerrojo_coords:
                for coord in self.coordenadas_doradas:
                    if self.estado_pepitas.get(coord, False):
                        self.estado_pepitas[coord] = False
                        coord_obj = coord
                        break

            if coord_obj is not None:
                x, y = coord_obj
                distancia = abs(x) + abs(y)
                tiempo_viaje = distancia * self.seg_simulado

                time.sleep(tiempo_viaje * 2)

                self._añadir_pepita()
                print("Pepita entregada, total en la base:", self.num_pepitas)

                Thread(
                    target=self._reactivar_pepita,
                    args=(coord_obj,),
                    daemon=True
                ).start()
            else:
                time.sleep(0.5)

    def añadir_recolectora(self):
        t = Thread(target=self._recolectora, daemon=True)
        t.start()
        self.recolectoras.append(t)
        return t

    def comprar_carretilla(self):
        if self._quitar_pepitas(3):
            self.sem_carretillas.release()
            print("Compra realizada: 1 carretilla nueva")
            return True
        print("No hay pepitas suficientes para comprar")
        return False

    def _distancia(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    def _calcular_ruta(self):
        while not self.stop_event.is_set():
            ruta = []

            with self.cerrojo_coords:
                disponibles = [
                    coord
                    for coord in self.coordenadas_doradas
                    if self.estado_pepitas.get(coord, False)
                ]

            if len(disponibles) < 3:
                time.sleep(1)
                continue

            candidatas = disponibles[:10]
            actual = (0, 0)

            while candidatas and len(ruta) < 5:
                siguiente = min(candidatas, key=lambda c: self._distancia(actual, c))
                ruta.append(siguiente)
                candidatas.remove(siguiente)
                actual = siguiente

            ruta_tuple = tuple(ruta)

            with self.cerrojo_rutas:
                if ruta and ruta_tuple not in self.rutas_registradas:
                    self.rutas_pendientes.append(ruta)
                    self.rutas_registradas.add(ruta_tuple)
                    print("Nueva ruta calculada:", ruta)

            time.sleep(1)

    def añadir_calculadora_ruta(self):
        t = Thread(target=self._calcular_ruta, daemon=True)
        t.start()
        self.calculadoras_ruta.append(t)
        return t

    def _recolectora_viajera(self):
        print("Recolectora viajera lista en la base")

        while not self.stop_event.is_set():
            self.sem_carretillas.acquire()

            if self.stop_event.is_set():
                self.sem_carretillas.release()
                return

            ruta = None

            with self.cerrojo_rutas:
                if self.rutas_pendientes:
                    ruta = self.rutas_pendientes.pop(0)
                    self.rutas_registradas.discard(tuple(ruta))

            if not ruta:
                self.sem_carretillas.release()
                time.sleep(1)
                continue

            actual = (0, 0)
            distancia_total = 0
            pepitas_recogidas = 0

            for coord in ruta:
                distancia_total += self._distancia(actual, coord)
                actual = coord

                recogida = False

                with self.cerrojo_coords:
                    if self.estado_pepitas.get(coord, False):
                        self.estado_pepitas[coord] = False
                        recogida = True

                if recogida:
                    pepitas_recogidas += 1
                    Thread(
                        target=self._reactivar_pepita,
                        args=(coord,),
                        daemon=True
                    ).start()

            distancia_total += self._distancia(actual, (0, 0))
            time.sleep(distancia_total * self.seg_simulado)

            dias_trabajados = max(1, math.ceil(distancia_total / 86400))
            salario = math.ceil(dias_trabajados / 10)

            with self.cerrojo_monedero:
                self.num_pepitas += pepitas_recogidas
                self.num_pepitas = max(0, self.num_pepitas - salario)

                print("Ruta completada:", ruta)
                print("Pepitas recogidas:", pepitas_recogidas)
                print("Días trabajados:", dias_trabajados)
                print("Salario pagado:", salario)
                print("Pepitas totales en la base:", self.num_pepitas)

            self.sem_carretillas.release()

    def añadir_recolectora_viajera(self):
        t = Thread(target=self._recolectora_viajera, daemon=True)
        t.start()
        self.recolectoras_viajeras.append(t)
        return t

    def _pagar_salarios(self):
        while not self.stop_event.is_set():
            time.sleep(30 * 86400 * self.seg_simulado)

            with self.cerrojo_monedero:
                salarios = len(self.exploradoras) + len(self.recolectoras)
                self.num_pepitas = max(0, self.num_pepitas - salarios)

                print("Salarios pagados:", salarios)
                print("Pepitas restantes:", self.num_pepitas)

    def añadir_pagador(self):
        t = Thread(target=self._pagar_salarios, daemon=True)
        t.start()
        return t

    def _gestor_compras(self):
        while not self.stop_event.is_set():
            with self.cerrojo_monedero:
                if self.num_pepitas >= 3:
                    self.num_pepitas -= 3
                    self.sem_carretillas.release()
                    print("Nueva carretilla comprada")

            time.sleep(5)

    def añadir_gestor_compras(self):
        t = Thread(target=self._gestor_compras, daemon=True)
        t.start()
        return t

    def estado_actual(self):
        with self.cerrojo_monedero, self.cerrojo_coords, self.cerrojo_rutas:
            return {
                "num_pepitas": self.num_pepitas,
                "coords_doradas": len(self.coordenadas_doradas),
                "rutas_pendientes": len(self.rutas_pendientes),
                "exploradoras": len(self.exploradoras),
                "recolectoras": len(self.recolectoras),
                "recolectoras_viajeras": len(self.recolectoras_viajeras),
            }

    def detener(self):
        self.stop_event.set()
        self.sem_carretillas.release()

import time

# Configuracion de la simulacion
# 30 dias simulados = 2,592,000 segundos
# Con factor 0.00001, la ejecucion real dura ~26 segundos
factor_velocidad = 0.00001
tiempo_limite_real = 25.92

central = CentralOperaciones(seg_simulado=factor_velocidad)

# 1. Generacion de recursos iniciales
# Exploradoras para encontrar pepitas y recolectoras basicas para el capital inicial
central.añadir_exploradora()
central.añadir_exploradora()
central.añadir_recolectora()
central.añadir_recolectora()

# 2. Activacion del sistema de recolectoras viajeras
# Calculan rutas optimas, gestionan el equipo (carretillas) y viajan
central.añadir_calculadora_ruta()
central.añadir_recolectora_viajera()
central.añadir_recolectora_viajera()
central.añadir_gestor_compras()

# 3. Control de gastos fijos
# Gestiona el pago de salarios de exploradoras y recolectoras clasicas cada 30 dias
central.añadir_pagador()

# Bucle de ejecucion por tiempo real equivalente a 30 dias simulados
inicio_simulacion = time.time()
try:
    while (time.time() - inicio_simulacion) < tiempo_limite_real:
        estado = central.estado_actual()
        print(f"Progreso 30 dias -> Pepitas: {estado['num_pepitas']} | "
              f"Viajeras activas: {estado['recolectoras_viajeras']} | "
              f"Rutas en cola: {estado['rutas_pendientes']}")
        time.sleep(5)
finally:
    # Finalizacion de todos los hilos
    central.detener()

print("\n--- Balance tras 30 dias de operacion ---")
resultado = central.estado_actual()
for clave, valor in resultado.items():
    print(f"{clave}: {valor}")

Recolectora clásica lista en la base
Recolectora clásica lista en la base
Recolectora viajera lista en la base
Recolectora viajera lista en la base
Progreso 30 dias -> Pepitas: 0 | Viajeras activas: 2 | Rutas en cola: 0
Pepita entregada, total en la base: 1
Pepita entregada, total en la base: 2
Pepita entregada, total en la base: 3
Pepita entregada, total en la base: 4
Pepita entregada, total en la base: 5
Pepita entregada, total en la base: 6
Pepita entregada, total en la base: 7
Pepita entregada, total en la base: 8
Pepita entregada, total en la base: 9
Pepita entregada, total en la base: 10
Pepita entregada, total en la base: 11
Nueva ruta calculada: [(4400, 98), (4654, 658), (4486, -743)]
Pepita entregada, total en la base: 12
Pepita entregada, total en la base: 13
Pepita entregada, total en la base: 14
Pepita entregada, total en la base: 15
Pepita entregada, total en la base: 16
Pepita entregada, total en la base: 17
Pepita entregada, total en la base: 18
Pepita entregada, total e

Pepita entregada, total en la base: 467
Pepita entregada, total en la base: 468
Salarios pagados: 4
Pepitas restantes: 464


## Texto para el notebook: decisiones de diseño e implementación

#### Decisiones de diseño e implementación

En esta práctica se ha desarrollado una simulación concurrente de una expedición de búsqueda y recolección de pepitas de oro. Para ello se han utilizado hilos, cerrojos y semáforos, con el objetivo de modelar de forma realista la exploración del mapa, la recogida de recursos y la coordinación entre distintos tipos de trabajadoras.

#### Exploración del mapa

Para la exploración se han utilizado hilos que representan a las exploradoras. Cada exploradora se encarga de revisar coordenadas del mapa para encontrar hashes dorados.
Una de las primeras decisiones de diseño ha sido evitar que dos exploradoras recorran las mismas coordenadas, ya que eso supondría una pérdida de tiempo y recursos. Para resolverlo, se ha usado un contador global de columnas (ultimo_x) protegido mediante un Lock. De esta forma, cada exploradora obtiene de forma segura una columna distinta del mapa y explora únicamente esa zona.

Esta solución permite repartir el trabajo de forma sencilla y evita exploración duplicada sin necesidad de estructuras más complejas.

#### Gestión de coordenadas doradas

Las coordenadas en las que se detecta una pepita potencial se almacenan en una lista compartida. Como varios hilos pueden acceder a esta estructura al mismo tiempo, se ha protegido con un cerrojo (cerrojoCoords) para evitar condiciones de carrera.

Además, para impedir que una misma coordenada se añada varias veces, se ha utilizado un conjunto auxiliar (set) con las coordenadas ya registradas. Esto mejora la consistencia de los datos y evita que las recolectoras trabajen varias veces sobre la misma coordenada detectada.

#### Recolectoras clásicas

Las recolectoras clásicas se han implementado también mediante hilos. Su comportamiento consiste en tomar una coordenada disponible, desplazarse hasta ella desde la base, regresar y añadir una pepita al monedero central.

El tiempo de desplazamiento se ha calculado usando la distancia de Manhattan, ya que el movimiento en el mapa se realiza coordenada a coordenada. Para simular el tiempo de viaje se ha usado sleep, ajustado a la escala temporal establecida en el enunciado.

Este diseño permite representar de manera sencilla el coste temporal de cada recolección y observar cómo las recolectoras clásicas dejan de ser eficientes a medida que las pepitas se encuentran más lejos de la base.

#### Reaparición de pepitas

Una vez recogida una pepita de una coordenada dorada, esa misma posición no vuelve a estar disponible inmediatamente, sino al cabo de un día simulado. Para modelar esto, se ha utilizado una estructura que guarda si cada pepita está disponible o no.

Cuando una recolectora recoge una pepita, esta pasa a estado no disponible. Después, se lanza un hilo que espera el tiempo correspondiente a un día simulado y vuelve a activar esa coordenada. De esta manera, se puede simular el reaparecimiento periódico del recurso sin bloquear el resto del sistema.

#### Recolectoras viajeras y rutas

Para mejorar la eficiencia, se han introducido recolectoras viajeras, capaces de recorrer varias coordenadas en una sola expedición antes de volver a la base. Esto reduce el número de viajes de ida y vuelta y aprovecha mejor los desplazamientos largos.

Como base para este comportamiento se ha implementado una calculadora de rutas. Esta función construye rutas sencillas usando una estrategia voraz inspirada en el problema del viajante. En lugar de buscar la mejor solución exacta, que sería muy costosa computacionalmente, se elige en cada paso la coordenada más cercana a la posición actual. Aunque no garantiza la ruta óptima, ofrece una solución razonable con un coste bajo y funciona bien para la simulación.

Las rutas calculadas se almacenan en una lista compartida protegida con su propio cerrojo, de forma que las recolectoras viajeras puedan tomarlas sin interferencias.

#### Uso del semáforo para las carretillas

El número de recolectoras viajeras trabajando al mismo tiempo está limitado por la cantidad de carretillas disponibles. Para representar este recurso se ha utilizado un Semaphore.

Cada vez que una recolectora viajera va a salir, debe hacer acquire() sobre el semáforo. Si no hay carretillas disponibles, queda bloqueada hasta que alguna vuelva a estar libre. Al regresar a la base, la recolectora libera la carretilla con release().

Esta decisión modela de forma natural el problema del acceso a recursos limitados y encaja muy bien con el uso típico de los semáforos en programación concurrente.

#### Monedero y salarios

El número de pepitas disponibles en la base se ha representado mediante una variable compartida. Como puede modificarse desde distintos hilos, por ejemplo al recoger pepitas, pagar salarios o comprar carretillas, se ha protegido con un cerrojo específico (cerrojoMonedero).

Se ha diferenciado entre dos sistemas salariales:

Las exploradoras y recolectoras clásicas cobran 1 pepita cada 30 días simulados.

Las recolectoras viajeras cobran al regresar a la base, según la duración de la ruta.

Para las trabajadoras mensuales se ha implementado un hilo independiente que espera 30 días simulados y descuenta del monedero el salario correspondiente. En cambio, el salario de las viajeras se calcula al finalizar cada ruta, ya que depende directamente del tiempo invertido en esa expedición.

#### Compra automática de material

También se ha añadido un gestor de compras, que permite adquirir carretillas nuevas cuando la central dispone de suficientes pepitas. Esta parte se ha automatizado para que la simulación evolucione sola y no dependa de intervención manual una vez iniciada.

Esto permite observar mejor la transición desde un sistema basado principalmente en recolectoras clásicas a otro con más peso de las recolectoras viajeras.

#### Optimización y organización general

A nivel general, una decisión importante ha sido utilizar cerrojos separados para cada recurso compartido importante, en lugar de un único cerrojo global. Por ejemplo, el mapa de coordenadas y el monedero tienen cerrojos distintos. Esto reduce bloqueos innecesarios y mejora la concurrencia, ya que varios hilos pueden trabajar sobre recursos diferentes al mismo tiempo.

También se ha preferido una solución progresiva y sencilla antes que una demasiado compleja. En una práctica como esta resulta más importante tener un sistema concurrente correcto, comprensible y funcional que una optimización extrema difícil de justificar.

#### Conclusión

En conjunto, el diseño intenta equilibrar simplicidad, corrección y aprovechamiento de la programación concurrente. Se han utilizado hilos para representar comportamientos simultáneos del sistema, Lock para proteger recursos compartidos y Semaphore para modelar recursos limitados.
Además, se ha incorporado una estrategia básica de planificación de rutas para mejorar la eficiencia de la recolección, manteniendo un coste computacional razonable.

El resultado es una simulación que reproduce varios problemas típicos de concurrencia, como la coordinación entre tareas, el acceso seguro a datos compartidos y la gestión de recursos limitados.